# Semana 13 - Validação eOtimização de Modelos (Titanic)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# Carregar Dataset
path_dataset = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'

df = pd.read_csv(path_dataset)

# Visualizar as primeiras linhas
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
# Separar as categorias entre 'Sobreviveu' e 'Faleceu'

X = df.drop('Survived', axis=1)
y = df['Survived']

print(X.shape)
print(y.shape)

(891, 11)
(891,)


In [ ]:
# Definir colunas numéricas e categóricas
col_num = ['Age', 'Fare']
col_cat = ['Pclass', 'Sex', 'SibSp', 'Parch', 'Embarked']

# Pipeline para variáveis numéricas: tratar nulos (média) e escalonar
pipeline_num = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Pipeline para variáveis categóricas: tratar nulos (mais frequente) e categorizar com one-hot
pipeline_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Unir ambos
processor = ColumnTransformer([
    ('num', pipeline_num, col_num),
    ('cat', pipeline_cat, col_cat)
])

In [ ]:
# Criar Pipeline completo: Preprocessamento + modelo

pipeline = Pipeline([
    ('processor', processor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Dividir treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

# Treino
pipeline.fit(X_treino, y_treino)

# Avaliar
y_pred = pipeline.predict(X_teste)
accuracy_standard = accuracy_score(y_teste, y_pred)
print(f'Acurácia: {accuracy_standard:.4f}')

Acurácia: 0.7933


Teste: Trocar estratégia de Média para Mediana

In [ ]:
# Criação do Pipeline mediana
pipeline_num_median = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

processor_median = ColumnTransformer([
    ('num', pipeline_num_median, col_num),
    ('cat', pipeline_cat, col_cat)
])

pipeline_median = Pipeline([
    ('processor', processor_median),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Treino e Avaliação
pipeline_median.fit(X_treino, y_treino)
y_pred = pipeline_median.predict(X_teste)
accuracy_median = accuracy_score(y_teste, y_pred)
print(f'Acurácia: {accuracy_median:.4f}')

Acurácia: 0.7933


Teste: Trocar Scaler para Min/Max

In [ ]:
# Criação do pipeline MinMax
pipeline_num_minmax = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', MinMaxScaler())
])

processor_num_minmax = ColumnTransformer([
    ('num', pipeline_num_minmax, col_num),
    ('cat', pipeline_cat, col_cat)
])

pipeline_minmax = Pipeline([
    ('processor', processor_num_minmax),
    ('classifier', LogisticRegression(max_iter=1000))
])


# Treino e Avaliação
pipeline_minmax.fit(X_treino, y_treino)
y_pred = pipeline_minmax.predict(X_teste)
accuracy_minmax = accuracy_score(y_teste, y_pred)
print(f'Acurácia: {accuracy_minmax:.4f}')

Acurácia: 0.7989


Teste 3: Pipeline com Árvore de decisão

In [ ]:
# Criação do Pipeline para o modelo de árvore de decisão
pipeline_num_tree = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

pipeline_cat_tree = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

processor_tree = ColumnTransformer([
    ('num', pipeline_num_tree, col_num),
    ('cat', pipeline_cat_tree, col_cat)
])

pipeline_tree = Pipeline([
    ('processor', processor_tree),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# treino e teste
pipeline_tree.fit(X_treino, y_treino)
y_pred = pipeline_tree.predict(X_teste)
accuracy_tree = accuracy_score(y_teste, y_pred)
print(f'Acurácia: {accuracy_tree:.4f}')
print(f'Diferença: {accuracy_tree - accuracy_standard: .4f}')

Acurácia: 0.7709
Diferença: -0.0223


Teste: Não realizar escalonamento

In [ ]:
# Pipeline sem escalonamento
pipeline_num_no_scale = Pipeline([
    ('imputer', SimpleImputer(strategy='mean'))
])

pipeline_cat_no_scale = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

processor_no_scale = ColumnTransformer([
    ('num', pipeline_num_no_scale, col_num),
    ('cat', pipeline_cat_no_scale, col_cat)
])

pipeline_no_scale = Pipeline([
    ('processor', processor_no_scale),
    ('classifier', LogisticRegression(max_iter=1000))
])


# Treino e teste
pipeline_no_scale.fit(X_treino, y_treino)
y_pred_no_scale = pipeline_no_scale.predict(X_teste)
accuracy_no_scale = accuracy_score(y_teste, y_pred_no_scale)
print(f'Acurácia: {accuracy_no_scale:.4f}')
print(f'Diferença: {accuracy_no_scale - accuracy_standard: .4f}')

Acurácia: 0.7989
Diferença:  0.0056


Teste: Ignorar tratamento de valores nulos

In [ ]:
# Criação das variáveis sem tratamento de nulos
X_nulls = X.dropna()
y_nulls = y[X_nulls.index]

X_treino_no_nulls, X_teste_no_nulls, y_treino_no_nulls, y_teste_no_nulls = train_test_split(X_nulls, y_nulls, test_size=0.2, random_state=42)

# Criação do pipeline sem o tratamento de nulos
processor_no_imputer = ColumnTransformer([
    ('num', StandardScaler(), col_num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), col_cat)
])

pipeline_no_imputer = Pipeline([
    ('processor', processor_no_imputer),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Treino e teste
pipeline_no_imputer.fit(X_treino_no_nulls, y_treino_no_nulls)
y_pred_no_imputer = pipeline_no_imputer.predict(X_teste_no_nulls)
accuracy_no_imputer = accuracy_score(y_teste_no_nulls, y_pred_no_imputer)
print(f'Acurácia: {accuracy_no_imputer:.4f}')
print(f'Diferença: {accuracy_no_imputer - accuracy_standard: .4f}')

Acurácia: 0.7568
Diferença: -0.0365


Perguntas:

1. Qual estratégia deu o melhor resultado? Por quê?

| Modelo | Acurácia |
|--------|----------|
| Regressão Logística (média) | 0.7933 |
| Regressão Logística (mediana) | 0.7989 |
| Regressão Logística (MinMaxScaler) | 0.7989 |
| Decision Tree | 0.7654 |
| Regressão Logística (sem escalonar) | 0.7989 |
| Regressão Logística (sem tratar nulos) | 0.7568 |

Os melhores resultados foram com a Regressão Logística usando a Mediana e a Regressao Logística ignorando escalonamento, isso ocorre porque com as iterações fixas em 1000, o modelo sem o escalonamento pôde explorar o espaço de parâmetros.

2. O que acontece se você NÃO escalonar os dados?

A acurácia sem escalonar ficou igual ao StandardScaler usando a mediana. Isso ocorreu pela característica do dataset, onde as colunas Age e Fare apesar de possuirem escalas diferentes, não foram diferentes o suficiente para desestabilizar o modelo sem usar o escalonamento.

3. O que acontece se você não tratar os valores nulos?

Obtivemos a menor acurácia dentre os testes, ou seja, ao não tratarmos valores nulos, perderemos performance em qualquer modelo.